# Proyecto estadísticas de YouTube Avances IV

**Semana 10 · 24 de octubre de 2026**

Cristian Camilo Alarcon Delgadillo · Bryan Arley Saenz Cortes · Jonathan David Lancheros Bello

Este notebook evalúa el artefacto congelado de Avances III y documenta resultados, retos y límites. La prueba de 374 videos ya se conocía en Avances II: se utiliza de forma retrospectiva, sin nueva selección. La separación por fecha de publicación es una prueba de estrés interna; no valida un pronóstico futuro.

[Repositorio](https://github.com/Camilo19P8/youtube-estadisticas-nlp) · Código desarrollado en VS Code.

In [ ]:
import io, os, sys, urllib.request, zipfile, platform
from pathlib import Path
SOURCE_COMMIT = 'eb793a5282a6b6b5f20f1d67cf48fa965d928135'
url = f'https://codeload.github.com/Camilo19P8/youtube-estadisticas-nlp/zip/{SOURCE_COMMIT}'
archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
workdir = Path('/content/avances_iv')
workdir.mkdir(exist_ok=True)
archive.extractall(workdir)
repo = workdir / archive.namelist()[0].split('/')[0]
os.chdir(repo)
sys.path.insert(0, str(repo))
print('Código fuente fijado:', SOURCE_COMMIT)
print('Python Colab:', platform.python_version())
import runpy
runpy.run_path('download_data.py', run_name='__main__')

## Protocolo final

Se conserva el modelo elegido en Avances III. La referencia HGB se ajusta solo con los 1.493 videos de desarrollo. Se calculan RMSE, MAE y R² en escala log y errores en vistas originales. El bootstrap mantiene predicciones fijas. No se hace una búsqueda de hiperparámetros con la prueba conocida.

In [ ]:
from final_evaluation import evaluate_final
from threadpoolctl import threadpool_limits
with threadpool_limits(limits=2):
    summary_iv = evaluate_final()

In [ ]:
import pandas as pd, json, numpy as np
from IPython.display import display
print('EVALUACIÓN RETROSPECTIVA: prueba conocida, no externa')
retrospective = pd.read_csv('results_iv/retrospective_metrics.csv')
display(retrospective.round(5))
print('ESTRÉS POR FECHA DE PUBLICACIÓN')
display(pd.read_csv('results_iv/publication_stress.csv').round(5))
print('VERIFICACIONES DE SOFTWARE')
display(pd.Series(json.loads(Path('results_iv/verification.json').read_text())))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(retrospective['model'], retrospective['rmse'], color='#406879')
axes[0].set_xlabel('RMSE log · evaluación retrospectiva')
pred = pd.read_csv('results_iv/retrospective_predictions.csv')
axes[1].scatter(pred.observed_log, pred.prediction_log, s=14, alpha=.6, color='#406879')
limits = [pred.observed_log.min(), pred.observed_log.max()]
axes[1].plot(limits, limits, 'k--', linewidth=1)
axes[1].set(xlabel='Observado ln(1 + vistas)', ylabel='Predicción congelada')
fig.tight_layout()
fig.savefig('results_iv/final_comparison.png', dpi=180)
plt.show()

## Interpretación y reproducción

Los comentarios aislados, las interacciones y el ensamble responden a entradas distintas. La precisión contemporánea no demuestra capacidad para anticipar vistas a siete o treinta días. Se necesitan canal, fecha de captura y observaciones posteriores para esa validación. La inferencia portable comprueba equivalencia numérica y validación de entradas; no es una nueva muestra predictiva.

El historial de GitHub documenta revisiones de esta cuenta; no se usa para atribuir aportes a otros integrantes. La evidencia colaborativa se distingue de los artefactos técnicos.

In [ ]:
import shutil
shutil.make_archive('/content/resultados_avances_iv', 'zip', 'results_iv')
print('Resultados guardados en /content/resultados_avances_iv.zip')